In [ ]:
import pytesseract
from pdf2image import convert_from_path
from jiwer import cer, wer
import pandas as pd
from pathlib import Path
from PIL import Image


TESSERACT_CONFIG = r'--oem 3 --psm 6 --dpi 200'

def normalize(text):
    return " ".join(text.lower().split())

# def ocr_pdf(pdf_path, dpi=200):
#     pages = convert_from_path(pdf_path, dpi=dpi)
#     extracted_text = ""
#     for page in pages:
#         text = pytesseract.image_to_string(page, config=TESSERACT_CONFIG)
#         extracted_text += text + "\n"
#     return extracted_text

def ocr_png(png_path):
    from PIL import Image
    image = Image.open(png_path)
    image.info['dpi'] = (200, 200)
    text = pytesseract.image_to_string(image, config=TESSERACT_CONFIG)
    return text

In [ ]:
dataset_path = Path("dataset-png") # Change to "dataset-pdf" if processing PDFs
results = []

for png_path in dataset_path.glob("*.png"): # Change to .pdf if processing PDFs
    gt_path = png_path.with_suffix(".txt")
    if not gt_path.exists():
        continue

    try:
        ground_truth = normalize(gt_path.read_text(encoding="utf-8"))
        ocr_text = normalize(ocr_png(png_path))
    except Exception as e:
        print(f"Error on {png_path.name}: {e}")
        continue

    results.append({
        "file": png_path.name,
        "CER": cer(ground_truth, ocr_text),
        "WER": wer(ground_truth, ocr_text)
    })

df = pd.DataFrame(results)

In [3]:
print("Average CER:", df["CER"].mean())
print("Average WER:", df["WER"].mean())
print(df.sort_values("WER", ascending=False))

Average CER: 0.10793473271187015
Average WER: 0.14045997180134767
                          file       CER       WER
61   motion_civil_clean_06.png  0.510774  0.544843
72   motion_civil_clean_01.png  0.502262  0.526559
63   motion_civil_clean_04.png  0.499659  0.524444
47   motion_civil_clean_09.png  0.502116  0.519630
58   motion_civil_clean_07.png  0.480743  0.511983
..                         ...       ...       ...
94    civil_order_clean_04.png  0.003155  0.030568
9     civil_order_clean_08.png  0.003193  0.030568
103   civil_order_clean_07.png  0.003183  0.030303
95    civil_order_clean_10.png  0.003183  0.030172
87    civil_order_clean_15.png  0.002527  0.021739

[108 rows x 3 columns]
